In [1]:
#load tools
import numpy as np
from scipy.integrate import solve_bvp
import matplotlib.pyplot as plt


In [7]:
#define constants
#molecular weight of relevant 
MW_Li6 = 6.0015
MW_Li7 = 7.0160
MW_Na = 22.9898
MW_K = 39.0983
MW_Mg = 24.305
MW_Mn = 54.9380
MW_Cl = 35.453
MW_SO4 = 96.06
MW_H2O = 18.01528
MW_NO3 = 62.0049
MW_OH = 17.0073
MW_H = 1.00784
#physiochemical constants
F = 96485.3329 #C/mol (Faraday's constant)
R = 8.314462618 #J/(mol*K) (Universal gas constant)
T = 298.15 #K (Temperature)
#charge numbers
z_Li6 = 1
z_Li7 = 1
z_Na = 1
z_K = 1
z_Mg = 2
z_Mn = 2
z_Cl = -1
z_SO4 = -2
z_H2O = 0
z_NO3 = -1
z_OH = -1
z_H = 1


In [8]:
#define parameters and boundary conditions
#diffusion coefficients - need to be updated with actual values
D_Li6 = 1.03e-9 #m^2/s (Diffusion coefficient of Li6)
D_Li7 = 1.03e-9 #m^2/s (Diffusion coefficient of Li7)
D_Na = 1.03e-9 #m^2/s (Diffusion coefficient of Na)
D_K = 1.03e-9 #m^2/s (Diffusion coefficient of K)
D_Mg = 1.03e-9 #m^2/s (Diffusion coefficient of Mg)
D_Mn = 1.03e-9 #m^2/s (Diffusion coefficient of Mn)
D_Cl = 1.03e-9 #m^2/s (Diffusion coefficient of Cl)
D_SO4 = 1.03e-9 #m^2/s (Diffusion coefficient of SO4)
D_H2O = 1.03e-9 #m^2/s (Diffusion coefficient of H2O)
D_NO3 = 1.03e-9 #m^2/s (Diffusion coefficient of NO3)
D_OH = 1.03e-9 #m^2/s (Diffusion coefficient of OH)
D_H = 1.03e-9 #m^2/s (Diffusion coefficient of H) 
#operational parameters
dphidx = -750 #V/m (Potential gradient)
L = 1e-4 #m (Membrane thickness?)  
C_left = 1000 #mol/m^3 (Concentration at the left boundary) (Feed)
C_right = 0 #mol/m^3 (Concentration at the right boundary) (Product)


In [9]:
#dictionary of molecular weights and charge numbers for easy access
element_data = {
    'Li6': {'MW': MW_Li6, 'z': z_Li6, 'D_aq': D_Li6},
    'Li7': {'MW': MW_Li7, 'z': z_Li7, 'D_aq': D_Li7},
    'Na': {'MW': MW_Na, 'z': z_Na, 'D_aq': D_Na},
    'K': {'MW': MW_K, 'z': z_K, 'D_aq': D_K},
    'Mg': {'MW': MW_Mg, 'z': z_Mg, 'D_aq': D_Mg},
    'Mn': {'MW': MW_Mn, 'z': z_Mn, 'D_aq': D_Mn},
    'Cl': {'MW': MW_Cl, 'z': z_Cl, 'D_aq': D_Cl},
    'SO4': {'MW': MW_SO4, 'z': z_SO4, 'D_aq': D_SO4},
    'H2O': {'MW': MW_H2O, 'z': z_H2O, 'D_aq': D_H2O},
    'NO3': {'MW': MW_NO3, 'z': z_NO3, 'D_aq': D_NO3},
    'OH': {'MW': MW_OH, 'z': z_OH, 'D_aq': D_OH},
    'H': {'MW': MW_H, 'z': z_H, 'D_aq': D_H}
}

In [12]:
#define ion of interest for the model
Ion_of_interest = 'Li7'


In [13]:
#pull from dictonary for ion of interest
MW_Ion = element_data[Ion_of_interest]['MW']
z_Ion = element_data[Ion_of_interest]['z']
D_Ion = element_data[Ion_of_interest]['D_aq']

In [ ]:
#calculate the hindered diffusion coefficient of an ion
def hindered_D_Ion(D_aq, ion_radius, pore_radius):
    """
    Calculate the hindered diffusion coefficient of an ion in a pore.

    Parameters:
    D_aq (float): Aqueous diffusion coefficient of the ion (m^2/s).
    ion_radius (float): Radius of the ion (m).
    pore_radius (float): Radius of the pore (m).

    Returns:
    float: Hindered diffusion coefficient (m^2/s).
    """
    # Calculate the hindrance factor using the Renkin equation
    hindrance_factor = (1 - (ion_radius / pore_radius))**2 * (1 - 2.104 * (ion_radius / pore_radius) + 2.09 * (ion_radius / pore_radius)**3)
    
    # Calculate the hindered diffusion coefficient
    D_hindered = D_aq * hindrance_factor
    
    return D_hindered

#alternattive approach to calculate the hindered diffusion coefficient of an ion

def local_diffusivity(D_aq, ion_radius=None, pore_radius=None):
    if pore_radius is None:
        return D_aq   # no confinement — bulk solution, current case
    lam = ion_radius / pore_radius
    return D_aq * (1 - lam)**2   # placeholder hindrance factor, upgrade later
#define D_effective (Actually used in the flux_ode function) for the ion of interest)
D_effective = local_diffusivity(element_data[Ion_of_interest]["D_aq"])   # pore_radius=None -> just returns D_aq

In [ ]:
#model steady state Nernst-Planck Flux for a single ion species across membrane only
def flux_ode(x, C, p):
    J = p[0]  # Flux (mol/m^2/s)
    dCdx=J/D_effective - z_Ion*F/(R*T)*C[0]*dphidx
    return np.vstack([dCdx])